# **Feature Exploration**

## Objectives

* Investigate whether domain-informed derived features improve the predictive power of the data

## Inputs

* "outputs/datasets/cleaned/HotelBookingsClean.csv"
* "outputs/correlation/RankedFeatures.csv" 

## Outputs

* Decisions about which derived features deserve further exploration

## Additional Comments

* This notebook extends the [correlation study](/jupyter_notebooks/05_correlation_study.ipynb) to explore *derived* features prior to formal feature engineering
* Candidate transformations were selected using hotel revenue management knowledge as well as statstical observations
* Every derived feature was evaluated using the same methodology as notebook 5 to maintain consistency
* Features are only retained if the improve predictive signal or offer a meaningful simplification for modelling

---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

current_dir = os.getcwd()
current_dir

## Load Data

* Load cleaned dataset

In [ ]:
import pandas as pd
df = pd.read_csv("outputs/datasets/cleaned/HotelBookingsClean.csv")
df.head(3)


In [ ]:
df.info()

* Re-run categorical type changes

In [ ]:
categorical_list = ["is_repeated_guest", "agent", "company"]
for col in categorical_list:
    df[col] = df[col].astype("category")

* Load ranked features

In [ ]:
ranked_features = pd.read_csv("outputs/correlation/RankedFeatures.csv")
ranked_features.sort_values(by=["feature_type", "rank_in_type"])

---

## Updated Feature Analysis

* Re-check feature distribution following cleaning

* Obtain list of numeric features

In [ ]:
def numeric_features(df):
    col_list = []
    for col in df.columns.to_list():
        if df[col].dtypes == "int64" or df[col].dtypes == "float64":
            col_list.append(col)
    return col_list       
        
col_list = numeric_features(df)
print(col_list)
print(f"There are {len(col_list)} numeric features")

* Plot numeric features

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math

def plot_numeric_features(cols):
    ncols = 4
    nrows = math.ceil(len(cols) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(15, 15))
    axs = axs.flatten()

    for i, col in enumerate(cols):
        sns.histplot(data=df,
                     x=col,
                     kde=True,
                     ax=axs[i])
        
    for j in range(len(cols), len(axs)):
        axs[j].set_visible(False)    

    plt.tight_layout()

plot_numeric_features(col_list)

* The post-cleaning distributions largely mirror the original EDA. `is_canceled` confirms the ~37% cancellation split. `lead_time` and `adr` remain right-skewed, with `adr` closer to a moderate skew and lead_time showing a much sharper, longer tail.
* `arrival_date_year` shows 3 discrete peaks reflecting the incomplete 2015/2017 years against the full 2016 year. `arrival_date_week_number` and `arrival_date_day_of_month` are both reasonably evenly spread, consistent with the earlier periodicity findings.
* `adults`, `children`, `babies`, `stays_in_weekend_nights` and `stays_in_week_nights` all show discrete, clustered values rather than continuous spread, reflecting their nature as small integer counts.
* `previous_cancellations`, `previous_bookings_not_canceled`, `days_in_waiting_list`, `required_car_parking_spaces` and `total_of_special_requests` all remain heavily zero-inflated, confirming these as strong candidates for binary or binned treatment rather than continuous use as-is.

* Separate long-tail distribution from outliers and create boxplots to examine further

In [ ]:
long_tail_cols = ["lead_time", "adr", "days_in_waiting_list",
                "previous_cancellations", "previous_bookings_not_canceled",
                "stays_in_weekend_nights", "stays_in_week_nights",
                "required_car_parking_spaces", "total_of_special_requests", "children", "babies"]

def plot_long_tail_features(cols):
    ncols = 3
    nrows = math.ceil(len(cols) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(12, 15))
    axs = axs.flatten()

    for i, col in enumerate(cols):
        sns.boxplot(data=df,
                     x=col,
                     ax=axs[i])   
        
    for j in range(len(cols), len(axs)):
        axs[j].set_visible(False) 

    plt.tight_layout()

plot_long_tail_features(long_tail_cols)

* All eleven features show extensive outlier flagging beyond the IQR whiskers, consistent with the zero-inflated distributions seen in the histograms. `lead_time`, `adr` and `stays_in_weekend_nights`/`stays_in_week_nights` show a genuine box-and-whisker shape with a long tail of outliers extending well beyond it, while the remaining features (`days_in_waiting_list`, `previous_cancellations`, `previous_bookings_not_canceled`, `required_car_parking_spaces`, `total_of_special_requests`, `children`, `babies`) are so heavily concentrated at zero that the box itself collapses to a thin line, with the "outliers" effectively representing the entire non-zero population rather than true anomalies.
* This distinction matters for treatment: features with a genuine long tail on top of a real distribution (`lead_time`, `adr` and `stays_in_weekend_nights`/`stays_in_week_nights`) are reasonable candidates for outlier capping, whereas the zero-collapsed features are better represented by presence/count-based treatments than by capping, since "outlier" here really just means "non-zero."

**Suggested Treatments for Feature Engineering**
* `lead_time`, `adr` and `stays_in_weekend_nights`/`stays_in_week_nights` show a genuine long tail beyond the IQR and are candidates for winsorizing. This is a standard preprocessing step and is not explored further in this notebook — it will be applied as part of the pipeline in [Feature Engineering](/jupyter_notebooks/07_feature_engineering.ipynb).
* `days_in_waiting_list`, `previous_cancellations`, `previous_bookings_not_canceled`, `required_car_parking_spaces`, `total_of_special_requests`, `children` and `babies` are all zero-collapsed rather than genuinely long-tailed, and are better suited to binary or binned treatment. These are explored as derived features below. 

---

## Investigate possible feature treatments

The previous 2 notebooks surfaced some features that might benefit from feature engineering as well as domain informed derived features. This notebook aims to explore the effects of the proposed treatments prior to formal feature engineering steps.

### Strategy

| Candidate(s) | Reason(s) | Transformation(s) |
| --- | --- | --- |
| adults, children, babies | Occupancy represented by 3 variables, family-type booking not well represented | total_guests unified feature, is_family binary flag |
| stays_in_week_nights, stays_in_weekend_nights | Length of stay represented by 2 variables, midweek/weekend not well represented | los unified feature, add is_weekend binary flag |
| arrival_date_year, arrival_date_month, arrival_date_week_number, arrival_date_day_of_month | Arrival date represented by 4 variables, cyclical calendar features not correctly represented | Drop arrival_date_year, sin/cos encoding on arrival_date_month and arrival_date_week_number, season time-bucketed column if seasonality is a predictor |
| lead_time | Current distribution heavily skewed | Group into time-buckets as explored in eda |
| previous_cancellations, previous_bookings_not_canceled | Customer history represented by 2 continuous numeric variables | Unify into has_cancelled_before binary flag |
| required_car_parking_spaces, total_of_special_requests | Additional needs represented by 2 continuous numeric variables | Unify into has_additional_needs binary flag |
| agent, company | Long-tail categorical variables with high zero values | Group into binary flags |
| country | Long-tail categorical variable | Group into domestic + international |
| days_in_waiting_list | Zero-inflated, heavily skewed continuous numeric variable | Replace with was_waitlisted binary flag |

* Generate results variable and validaton function

In [ ]:
results = []

In [ ]:
# Carry the threshold values from 05_correlation_study as a starting point
threshold = 0.1
pps_threshold = 0

In [ ]:
import ppscore as pps
from pandas.api.types import is_numeric_dtype

def feature_comparison(feature, target, raw):    
    
    def is_raw():
        if feature.name in raw.columns:
            return "Raw"
        else:
            return "Engineered"

    if is_numeric_dtype(feature):    
        def pearson():
            return feature.corr(target, method="pearson")

        pearson_score = pearson()

        def spearman():
            return feature.corr(target, method="spearman")

        spearman_score = spearman()
    else:
        pearson_score = "n/a"
        spearman_score = "n/a"
    
    def pps_score():
        pps_df = raw.copy()
        pps_df[target.name] = pps_df[target.name].astype("category")
        pps_df[feature.name] = feature
        return pps.score(pps_df, x=feature.name, y=target.name, sample=None)["ppscore"]
    
    pps_result = pps_score()
    
    def decision():
        pearson_yes = pearson_score != "n/a" and abs(pearson_score) > threshold
        spearman_yes = spearman_score != "n/a" and abs(spearman_score) > threshold
        
        if pearson_yes or spearman_yes or pps_result > pps_threshold:
            return "Yes"
        else:
            return "No"
        
    return {"Feature": feature.name,
            "Raw/Engineered": is_raw(),
            "Pearson": pearson_score,
            "Spearman": spearman_score,
            "PPS": pps_result,
            "Above Threshold": decision()}

* Populate `results` with raw features

In [ ]:
target = df["is_canceled"]

for col in df.columns:
    if col != "is_canceled":
        results.append(feature_comparison(df[col], target, df))

results

**Guest Composition**

* Assess whether an is_family binary flag is of more predictive value than guest counts alone

* Create binary "is_family" flag

In [ ]:
is_family = pd.Series(df[["children", "babies"]].gt(0).any(axis=1), name="is_family").astype("category")
is_family.head()

* Assess the predictive value of "is_family"

In [ ]:
is_family_val = feature_comparison(is_family, target, df)
is_family_val

* The binary flag did not clear the thresholds

* Create "total_guests" count from `adults`, `children` and `babies`

In [ ]:
total_guests = pd.Series(df["adults"] + df["children"] + df["babies"], name="total_guests")
total_guests.head()

* Investigate whether combining `adults`, `children` and `babies` into a total guest count provides a stronger predictive signal

In [ ]:
total_guests_val = feature_comparison(total_guests, target, df)
total_guests_val

* Total guests did not clear threshold

* Add results to results list

In [ ]:
results.append(is_family_val)
results.append(total_guests_val)

* Neither engineered feature improved the predictive signal relative to the original variables. The original guest compostition features will be retained for modelling

**Stay Characteristics**

* Evaluate whether combining `stays_in_week_nights` and `stays_in_weekend_nights` into a total length of stay (LOS) improves predictive performance

* Create Length of stay feature from `stays_in_week_nights` and `stays_in_weekend_nights`

In [ ]:
los = pd.Series(df["stays_in_week_nights"] + df["stays_in_weekend_nights"], name="LOS")
los.head()

* Evaluate whether the combined length of stay feature has any predictive signal

In [ ]:
los_val = feature_comparison(los, target, df)
los_val

* This feature did not reach the threshold

* Create binary "is_weekend" flag

In [ ]:
is_weekend = pd.Series(df["stays_in_weekend_nights"].gt(0), name="is_weekend").astype("category")
is_weekend.head()

* Assess predictive performance

In [ ]:
is_weekend_val = feature_comparison(is_weekend, target, df)
is_weekend_val

* The binary flag did not improve the feature's performance

* Add to results list

In [ ]:
results.append(los_val)
results.append(is_weekend_val)

* Combining weekday and weekend stays into a single length-of-stay feature did not improve predictive performance, neither did replacing `stays_in_weekend_nights` with an "is_weekend" flag. The original values are retained

**Temporal Variables**

* Use cyclical encoding on `arrival_date_month` and `arrival_date_week_number` to explore the effect of seasonality

* Create numeric categories for the months

In [ ]:
month_map = {"January": 1, "February": 2, "March": 3, "April": 4,
             "May": 5, "June": 6, "July": 7, "August": 8,
             "September": 9, "October": 10, "November": 11, "December": 12}
month = df["arrival_date_month"].map(month_map)
month.head()



* Use CyclicalFeatures to encode the month categories

In [ ]:
from feature_engine.creation import CyclicalFeatures
import numpy as np

month = month.to_frame()
cyclical = CyclicalFeatures(variables=None, drop_original=False)
month_cycle = cyclical.fit_transform(month)
month_cycle.head()

* Create dataframe to test performance via pps

In [ ]:
month_test = pd.concat([target, month_cycle], axis=1)
month_test.head()

* Apply pps test

In [ ]:
month_test["is_canceled"] = month_test["is_canceled"].astype("category")
month_pps = pps.predictors(month_test, y="is_canceled")
month_pps

* Cyclical encoding on `arrival_date_month` did not produce a pps score

* Apply cyclical encoding to `arrival_date_week_number`

In [ ]:
week = df[["arrival_date_week_number"]]

cyclical = CyclicalFeatures(variables=None, drop_original=False, max_values={"arrival_date_week_number": 52})
week_cycle = cyclical.fit_transform(week)
week_cycle.head()

* Create dataframe to test performance via pps

In [ ]:
week_test = pd.concat([target, week_cycle], axis=1)
week_test.head()

* Apply pps test

In [ ]:
week_test["is_canceled"] = week_test["is_canceled"].astype("category")
week_pps = pps.predictors(week_test, y="is_canceled")
week_pps

* Cyclical encoding on `arrival_date_week_number` did not produce a pps score

* Cyclical encoding of `arrival_date_month` and `arrival_date_week_number` did not improve predictive performance, indicating that seasonal effects are not a major driver of booking cancellation within this dataset. Seasonal time-bucketing will not be performed
* `arrival_date_year` will be excluded from modelling to avoid introducing dataset-specific temporal patterns

**Booking Behaviour**

* Explore whether lead time benefits from transformation or categorisation

* Create the lead time categories used in previous notebooks

In [ ]:
bins = [-np.inf, 7 ,30 ,90, np.inf]
lead_time_cat = pd.cut(df["lead_time"], bins, labels=["Last Minute", "Short Range", "Mid Range", "Long Range"])
lead_time_cat.name = "lead_time_category"
lead_time_cat.head()

* Assess the predictive value of the bucketed feature

In [ ]:
lead_time_val = feature_comparison(lead_time_cat, target, df)
lead_time_val

* Categorising `lead_time` into booking windows produced a minor reduction in PPS from 0.25 to 0.23

* Consider whether previous history can be better represented through derived behavioural features

* Create "has_canceled_before" flag

In [ ]:
has_canceled_before = pd.Series(df["previous_cancellations"].gt(0)).astype("category")
has_canceled_before.name = "has_canceled_before"
has_canceled_before.head()

* Evaluate the predictive performance

In [ ]:
has_canceled_before_val = feature_comparison(has_canceled_before, target, df)
has_canceled_before_val

* The binary representation caused almost no improvement in PPS. Both representations could be considered during modelling

* Combine `total_of_special_requests` and `required_car_parking_spaces` into a "has_additional_needs" flag

In [ ]:
has_additional_needs = pd.Series(
    df[["total_of_special_requests", "required_car_parking_spaces"]].gt(0).any(axis=1),
    name="has_additional_needs"
).astype("category")
has_additional_needs.head()

* Evaluate the predictive performance

In [ ]:
has_additional_needs_val = feature_comparison(has_additional_needs, target, df)
has_additional_needs_val

* Neither of the raw features produced a pps score individually so there is a real improvement there, however the lack of pearson/spearman in the binary feature means that there can be no absolute comparison. Both versions could be considered at modelling stage.

* Add to results

In [ ]:
results.append(lead_time_val)
results.append(has_canceled_before_val)
results.append(has_additional_needs_val)

**Long-tail compression**

* From the RankedFeatures table we can see that `country` and `agent` already have predictive signal. `company`has previously shown none but also has a heavily zero-inflated skew, now we will consider if compiling them into top_n + other improves these results.

In [ ]:
def plot_categoric_features(cols):
    ncols = 1
    nrows = 3
    fig, axs = plt.subplots(nrows, ncols, figsize=(15, 18))
    axs = axs.flatten()

    for i, col in enumerate(cols):
        order = df[col].value_counts().index

        sns.countplot(data=df,
                     x=col,
                     order=order,
                     ax=axs[i])
        
        axs[i].set_title(df[col].name)
    
    plt.tight_layout()
    plt.show()

cols = ["company", "agent", "country"]

plot_categoric_features(cols)


* All 3 features have such high cardinality that the plots are unreadable.

* All 3 appear as though a binary treatment may be more appropriate than binning. `country` will be "Domestic (PRT)" vs "International", `company` and `agent` will be simply "has_company" or "has_agent"

* Create binary flag for `country`

In [ ]:
is_domestic = pd.Series(df["country"].eq("PRT"), name="is_domestic").astype("category")
is_domestic.head()

* Evaluate the performance of "is_domestic"

In [ ]:
is_domestic_val = feature_comparison(is_domestic, target, df)
is_domestic_val

* This produced a minor negative effect on pps score (0.3287 -> 0.3241), the original feature should be retained.

* Create "has_agent" flag

In [ ]:
has_agent = pd.Series(df["agent"].astype("int64").gt(0), name="has_agent").astype("category")
has_agent.head()

* Evaluate the predictive performance of the flag

In [ ]:
has_agent_val = feature_comparison(has_agent, target, df)
has_agent_val

* This has removed all predictive signal that `agent` had previously, the original feature should be retained

* Create "has_company" flag

In [ ]:
has_company = pd.Series(df["company"].astype("int64").gt(0), name="has_company").astype("category")
has_company.head()

* Evaluate the performance of the feature

In [ ]:
has_company_val = feature_comparison(has_company, target, df)
has_company_val

* This also produced no predictive signal

* Add to results

In [ ]:
results.append(is_domestic_val)
results.append(has_agent_val)
results.append(has_company_val)

**Waitlist**

* `days_in_waiting_list` is a long-tail continuous numeric that may benefit from a binary `was_waitlisted` flag

* Create "was_waitlisted" flag

In [ ]:
waitlist = pd.Series(df["days_in_waiting_list"].gt(0)).astype("category")
waitlist.name = "was_waitlisted"
waitlist.head()

* Evaluate the performance

In [ ]:
waitlist_val = feature_comparison(waitlist, target, df)
waitlist_val

* The binary transformation did not improve predictive performance, the original feature will be retained.

* Add to results

In [ ]:
results.append(waitlist_val)

In [ ]:
results_df = pd.DataFrame(results)
results_features = results_df[results_df["Above Threshold"] == "Yes"]
results_features.sort_values(by="PPS", ascending=False)

---

## Summary

Across guest composition, stay characteristics, and temporal variables, no engineered feature outperformed its raw counterpart — `is_family`, `total_guests`, `LOS`, `is_weekend`, and cyclical encoding of `arrival_date_month`/`arrival_date_week_number` all failed to clear the predictive threshold.

In three cases the engineered version performed *worse* than the raw feature it was derived from: `lead_time_category` reduced PPS from 0.25 to 0.23, `is_domestic` reduced PPS from 0.3287 to 0.3241, and `has_agent` removed the predictive signal `agent` carried entirely. This indicates that granularity in `lead_time`, `country`, and `agent` is itself informative and should not be simplified away.

`has_additional_needs` is the one exception, returning a PPS of 0.137 where neither raw feature (`total_of_special_requests`, `required_car_parking_spaces`) scored individually. This can't be treated as a confirmed improvement, however, since PPS is the only metric available for categorical features — there's no Pearson/Spearman equivalent to cross-check against, unlike every other comparison in this notebook. It is carried forward as a candidate rather than a settled decision.

`has_company` and `was_waitlisted` showed no meaningful predictive signal either way.

---

## Conclusions

* Domain-informed feature transformations did not, on the whole, improve predictive signal — the raw dataset's granularity outperforms most simplified versions tested.
* Where a raw feature already carries individual-level signal (`lead_time`, `country`, `agent`), collapsing it into buckets or binary flags actively loses information rather than clarifying it.
* `has_additional_needs` is the sole candidate showing a possible improvement, though this is not conclusively comparable to its raw counterparts given the PPS-only metric limitation for categorical features. It is retained as an open candidate for the modelling stage rather than adopted outright.
* `has_company` returning no predictive signal reinforces the decision (made on missingness grounds) to drop `company` in the preprocessing pipeline.
* The original, raw feature set is retained as the baseline entering [Feature Engineering](/jupyter_notebooks/07_feature_engineering.ipynb); no engineered replacements from this notebook are adopted as default transformations.

## Next Steps

* Finalise preprocessing pipeline using the raw feature set, applying standard steps (winsorizing, imputation, encoding) as defined in [Feature Engineering](/jupyter_notebooks/07_feature_engineering.ipynb).
* Reassess `has_additional_needs` at modelling stage, where model-level comparison (rather than PPS/correlation alone) can properly evaluate it against the two raw features.
* Train and compare classification models using the validated feature set.
* Evaluate feature importance after modelling.